## Multiple Contracts

To demonstrate how we can use multiple contracts, we will create a strategy that looks at overnight price gaps for 3 contracts
It chooses the name that has the largest percentage overnight gap and goes long that name. The strategy exits by the end of the day
If price moves against us, we use a stop loss to get out. We use 1 minute bars for AAPL, NVDA and IBM

In [ ]:
# %%checkall
from collections import defaultdict
import polars as pl
import gambit as pq
import numpy as np
from dataclasses import dataclass

_logger = pq.get_child_logger(__name__)


@dataclass
class OvernightReturn:
    name: str
    on_ret: float  # overnight return
    timestamps: np.ndarray
    prices: np.ndarray

        
def create_price_dataframe(on_rets: dict[np.datetime64, OvernightReturn]) -> pl.DataFrame:
    dfs: list[pl.DataFrame] = []
    for date, on_ret in on_rets.items():
        df = pl.DataFrame({'timestamp': on_ret.timestamps.astype('M8[ns]'), 'price': on_ret.prices}).with_columns(pl.col('timestamp').dt.date().alias('date'), pl.lit(on_ret.name).alias('contract'))
        dfs.append(df)
    prices = pl.concat(dfs)
    prices = prices.select('timestamp', 'contract', 'date', 'price')  # re-order columns
    prices = prices.sort(['timestamp'])
    return prices
 

def create_overnight_returns(contracts: list[str]) -> dict[np.datetime64, OvernightReturn]:
    on_rets: dict[np.datetime64, OvernightReturn] = {}
    for name in contract_names:
        filename = pq.find_in_subdir('.', f'{name}.csv.gz')
        prices = pl.read_csv(filename, columns=['timestamp', 'c'], try_parse_dates=True).with_columns(pl.col('timestamp').dt.date().alias('date'))
        prices = prices.with_columns(pl.when(pl.col('date') > pl.col('date').shift(1)).then(pl.col('c') / pl.col('c').shift(1) - 1).otherwise(None).alias('on_ret'))
        date_rets = prices.filter(pl.col('on_ret').is_finite() & (pl.col('on_ret') > 0))
        dates = date_rets['date'].to_numpy().astype('M8[D]')
        on_ret = date_rets['on_ret'].to_numpy()
        for i, date in enumerate(dates):
            if date not in on_rets or on_ret[i] > on_rets[date].on_ret:
                date_prices = prices.filter(pl.col('date') == date)
                on_rets[date] = OvernightReturn(name, on_ret[i], date_prices['timestamp'].to_numpy().astype('M8[m]'), date_prices['c'].to_numpy())
    return on_rets


def add_signals(prices: pl.DataFrame) -> pl.DataFrame:
    first = prices.sort('timestamp').unique(subset=['contract', 'date'], keep='first').select('timestamp', 'contract').with_columns(pl.lit(True).alias('enter'))
    prices = prices.join(first, on=['timestamp', 'contract'], how='left').with_columns(pl.col('enter').fill_null(False), (pl.col('date').shift(-2) > pl.col('date')).fill_null(False).alias('eod'))   
    return prices


def create_price_function(on_rets: dict[np.datetime64, OvernightReturn]) -> pq.PriceFunctionType:
    price_dict: dict[str, tuple[np.ndarray, np.ndarray]] = {}
    for on_ret in on_rets.values():
        name = on_ret.name
        if name in price_dict:
            ts, pr = price_dict[name]
            ts = np.concatenate([ts, on_ret.timestamps])
            pr = np.concatenate([pr, on_ret.prices])
            price_dict[on_ret.name] = (ts, pr)
        else:
            price_dict[on_ret.name] = (on_ret.timestamps, on_ret.prices)
    for name, (timestamps, prices) in price_dict.items():
        order = np.argsort(timestamps, kind='stable')
        timestamps, prices = timestamps[order], prices[order]
        keep = np.concatenate(([True], timestamps[1:] != timestamps[:-1]))
        price_dict[name] = (timestamps[keep], prices[keep])
    price_function = pq.PriceFuncArrayDict(price_dict)
    return price_function

def stop_return_func(*args, **kwargs) -> float:
    return -0.01


@dataclass
class ContractFilter:
    '''
    For each day we want to trade only one symbol. So don't allow trade entry for all others
    '''
    def __init__(self, entry_contracts: dict[np.datetime64, list[str]]) -> None:
        self.entry_contracts = entry_contracts
          
    def __call__(self, _, i: int, timestamps: np.ndarray, *args, **kwargs) -> list[str]:
        date = timestamps[i].astype('M8[D]')
        return self.entry_contracts[date]


def post_trade(trade: pq.Trade, context: pq.StrategyContextType) -> None:
    '''
    Add entry price to context so we can use it for stops
    '''
    if not hasattr(context, 'entry_prices'):
        context.entry_prices = defaultdict(dict)
    date = trade.timestamp.astype('M8[D]')
    context.entry_prices[date][trade.order.contract.symbol] = trade.price
    

if __name__ == '__main__':
    contract_names = ['AAPL', 'NVDA', 'IBM']
    on_rets: dict[np.datetime64, OvernightReturn] = create_overnight_returns(contract_names)
    prices = create_price_dataframe(on_rets)
    prices = add_signals(prices)

    # add the stop price so we can refer to it in
    price_function = create_price_function(on_rets)
    # stop_return_function = stop_return_func  # create_stop_return_function(on_rets)
    prices = prices.with_columns(pl.lit(True).alias('stop'))
    
    entry_contracts = {date: [on_ret.name] for date, on_ret in on_rets.items()}
    
    # This rule allows us to enter trades and get out with a limited loss when a stop is hit.
    entry_rule = pq.BracketOrderEntryRule(
        reason_code='OVERNIGHT_RETURN',  # useful for reporting when we have multiple entry conditions
        contract_filter=ContractFilter(entry_contracts),
        price_func=price_function, 
        # stop price is used for position sizing.  Also, we will not enter if the price is already below 
        # stop price for long trades and vice versa
        stop_return_func=stop_return_func,
        long=True,  # whether we enter a long or short position
        percent_of_equity=0.1,  # set the position size so that if the stop is hit, we lose no more than this
        single_entry_per_day=True)  # if we are stopped out, do we allow re-entry later in the day

    # ClosePositionExitRule fully exits a position using either a market or limit order
    # In this case, we want to exit at EOD so we are flat overnight
    exit_rule_stop = pq.StopReturnExitRule(
        reason_code='STOPPED_OUT',
        price_func=price_function,
        stop_return_func=stop_return_func)

    # Exit when the stop price is reached
    exit_rule_eod = pq.ClosePositionExitRule(
        reason_code='EOD',
        price_func=price_function)
    
    strat_builder = pq.StrategyBuilder(data=prices)
    for contract in contract_names:
        strat_builder.add_contract(contract)
    strat_builder.set_price_function(price_function)
    

    # Setup the rules we setup above so they are only called when the columns below in our data dataframe are true
    strat_builder.add_series_rule('enter', entry_rule, position_filter='zero')
    strat_builder.add_series_rule('eod', exit_rule_eod, position_filter='positive')
    strat_builder.add_series_rule('stop', exit_rule_stop, position_filter='positive')
    
    market_sim = pq.SimpleMarketSimulator(price_func=price_function)
    market_sim.post_trade_func = post_trade
    market_sim.slippage_pct = 0.005  # half a percent each way
    strat_builder.add_market_sim(market_sim)
    strat_builder.set_log_orders(True)

    # create the strategy and run it
    strategy = strat_builder()
    strategy.run()

In [ ]:
strategy.df_roundtrip_trades()

In [ ]:
strategy.evaluate_returns(periods_per_year=252, plot=pq.has_display());